# Utils

In [ ]:
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return ""

    s = str(s)

    # fix common mojibake
    s = s.replace("‚Äôs", "’s")
    s = s.replace("‚Äô", "’")
    s = s.replace("‚Äù", '"')
    s = s.replace("‚Äú", '"')

    # normalize unicode
    s = unicodedata.normalize("NFKC", s)

    # normalize apostrophes to ASCII
    s = s.replace("’", "'")

    return s.strip()

# Recall@K

In [ ]:
import pandas as pd

retrieval_file = ""
groundtruth_file = ""

df_ret = pd.read_csv(retrieval_file)
df_gt = pd.read_csv(groundtruth_file)

print("Retrieval shape:", df_ret.shape)
print("Ground truth shape:", df_gt.shape)

df_ret.head(), df_gt.head()
ground_truth = {}

for _, row in df_gt.iterrows():
    t1 = str(row["Text1"]).strip()
    t2 = str(row["Text2"]).strip()

    ground_truth.setdefault(t1, set()).add(t2)
    ground_truth.setdefault(t2, set()).add(t1)

print("Total anchors with ground truth:", len(ground_truth))

results = []

for (pipeline, k), group in df_ret.groupby(["Pipeline", "Top_K"]):

    hits = 0
    total = 0

    for anchor, g in group.groupby("Anchor_text"):
        anchor = str(anchor).strip()

        if anchor not in ground_truth:
            continue

        total += 1

        retrieved = (
            g.sort_values("Rank")["Requirement"]
            .astype(str)
            .str.strip()
            .tolist()
        )

        if any(r in ground_truth[anchor] for r in retrieved):
            hits += 1

    recall = hits / total if total > 0 else 0

    results.append({
        "Pipeline": pipeline,
        "K": k,
        "Hits": hits,
        "Total_queries": total,
        "Recall@K": recall
    })

recall_df = pd.DataFrame(results).sort_values(["Pipeline", "K"])
recall_df


Retrieval shape: (265891, 7)
Ground truth shape: (10626, 4)
Total anchors with ground truth: 783


,Pipeline,K,Hits,Total_queries,Recall@K
0,KAG,1,390,651,0.599078
1,KAG,2,481,651,0.738863
2,KAG,3,511,651,0.784946
3,KAG,5,540,651,0.829493
4,KAG,10,592,651,0.909370
5,KAG,15,623,651,0.956989
6,KAG,20,634,651,0.973886
7,KAG,25,637,651,0.978495
8,KAG,30,642,651,0.986175
9,KAG,35,645,651,0.990783
